In [1]:
%pip install pandas
%pip install sodapy


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sodapy import Socrata

# Unauthenticated client only works with public data sets. Note 'None'
# in place of application token, and no username or password:
#client = Socrata("data.transportation.gov", None)

client = Socrata( "data.transportation.gov",
                     '1hCte9vYL7IpE7pFTGR1cbn8y',
                     username="hugo.baca4515@coyote.csusb.edu",
                     password="Hugobacaalex1!")

# First 2000 results, returned as JSON from API / converted to Python list of
# dictionaries by sodapy.
results = client.get_all("7wn6-i5b9")

# Convert to pandas DataFrame
Raw_2023 = pd.DataFrame.from_records(results)
#let's see what we get back
print(Raw_2023)


       railroadcode                        railroadname  year incidentnumber  \
0                UP      Union Pacific Railroad Company  2009      0809LV010   
1                UP      Union Pacific Railroad Company  2009      0809PR005   
2                UP      Union Pacific Railroad Company  2009      0809RS016   
3                UP      Union Pacific Railroad Company  2009      0809SA011   
4                UP      Union Pacific Railroad Company  2009      0809SL005   
...             ...                                 ...   ...            ...   
244704         CRSH       Consolidated Rail Corporation  2016         123182   
244705         CRSH       Consolidated Rail Corporation  2016         123328   
244706         NJTR  New Jersey Transit Rail Operations  2018      201802077   
244707         NJTR  New Jersey Transit Rail Operations  2018      201807361   
244708         NJTR  New Jersey Transit Rail Operations  2018      201807366   

       incidentyear incidentmonth maint

In [3]:
from numpy.core.fromnumeric import sort

column_headers = list(Raw_2023.columns.values)

column_headers.sort()

print("The Column Headers:")
for header in column_headers:
  print(header)

The Column Headers:
ampm
cityname
countycode
countyname
crossingilluminated
crossingusersinjured
crossinguserskilled
crossingwarning
crossingwarningcode
crossingwarningexpanded1
crossingwarningexpanded2
crossingwarningexpanded3
crossingwarningexpanded4
crossingwarningexpanded5
crossingwarningexpanded6
crossingwarningexpanded7
crossingwarningexpanded8
crossingwarningexpandedcode1
crossingwarningexpandedcode2
crossingwarningexpandedcode3
crossingwarningexpandedcode4
crossingwarningexpandedcode5
crossingwarningexpandedcode6
crossingwarningexpandedcode7
crossingwarningexpandedcode8
crossingwarningexplanation
crossingwarningexplanationcode
crossingwarninglocation
crossingwarninglocationcode
date
day
district
division
drivercondition
driverconditioncode
driverinvehicle
driverpassedvehicle
employeesinjured
employeeskilled
equipmentinvolved
equipmentinvolvedcode
equipmentstruck
equipmentstruckcode
equipmenttype
equipmenttypecode
estimatedrecordedspeedcode
estimatedvehiclespeed
form54filed
grad

In [ ]:
from numpy import NaN
dictionary_data_raw = pd.read_excel('../Data_Dictionaries/Form_54_Data_Dictionary.xlsx')

dictionary = pd.DataFrame.from_records(dictionary_data_raw)

dictionary.columns.values

dictionary = dictionary.filter(items=['Display Name', 'Description'])

print(dictionary)

In [12]:
# Create a dictionary to store the corresponding names and descriptions
name_to_description = {}
for index, row in dictionary.iterrows():
    name_to_description[row['Display Name']] = row['Description']

# Get the description for the variable 'report year'
description = name_to_description['Year']
print(description)

#Personal Prefence: Allows me to quickly pull descriptions for variable.

Report year 


In [21]:
#i would like to break up data into smaller chunks to make it easier to work with
# such as by 10 years

print(Raw_2023['year'].max())
print(Raw_2023['year'].min())




2023
1975


In [22]:
#assuming data collection gets better as time ages, 2023 > 2022 > ...
#so we will use 2023 as the last year of data collection
Raw_2023 = Raw_2023[Raw_2023['year'] == '2023']
print(Raw_2023.head())

from tabulate import tabulate
print(tabulate(Raw_2023, headers = 'keys', tablefmt = 'psql'))


      railroadcode                    railroadname  year incidentnumber  \
2529           CSX              CSX Transportation  2023      000211692   
2530          CFRC   Central Florida Rail Corridor  2023          23132   
15747           UP  Union Pacific Railroad Company  2023      0723LA001   
15748         BNSF            BNSF Railway Company  2023      SW0723201   
15749         BNSF            BNSF Railway Company  2023      SW0723203   

      incidentyear incidentmonth maintenancerailroadcode  \
2529            23            06                     CSX   
2530            23            05                    CFRC   
15747           23            07                      UP   
15748           23            07                    BNSF   
15749           23            07                    BNSF   

              maintenancerailroadname maintainenanceincidentnumber  \
2529               CSX Transportation                    000211692   
2530    Central Florida Rail Corridor           

In [26]:
#Explore for NA's, missing data, and redunnacy

isnaValues = Raw_2023.isna()
print(isnaValues)

# Check for NA values
missing_values = Raw_2023.isna().sum()

#check for null values
missing_data = Raw_2023.isnull().sum()
print(missing_data)

# Check for missing data
for column in Raw_2023.columns:
    if missing_values[column] > 0:
        print(f"The column {column} has {missing_values[column]} missing values.")

# Check for redundancy
for column in Raw_2023.columns:
    if Raw_2023[column].duplicated().sum() > 0:
        print(f"The column {column} has {Raw_2023[column].duplicated().sum()} duplicate values.")

#lets try to clean up raw data

# Calculate the percentage of missing values in each column
missing_values = Raw_2023.isna().mean()

# Drop columns with more than 30% missing values
drop_list = [column for column in missing_values.index if missing_values[column] > 0.3]

# Drop the columns
Raw_2023_clean = Raw_2023.drop(columns=drop_list)

print(f"\n The Raw Data Frame has {len(Raw_2023.columns)} labels.")
print(f"\n The Clean Data Frame now has {len(Raw_2023_clean.columns)} labels.")
print("\n Count of labels dropped: \n", len(Raw_2023.columns) - len(Raw_2023_clean.columns) )
print("\n Deleted Labels: \n", drop_list)
print("\n Cleaned Data Labels: \n", Raw_2023_clean.columns.values)

        railroadcode  railroadname   year  incidentnumber  incidentyear  \
2529           False         False  False           False         False   
2530           False         False  False           False         False   
15747          False         False  False           False         False   
15748          False         False  False           False         False   
15749          False         False  False           False         False   
...              ...           ...    ...             ...           ...   
233844         False         False  False           False         False   
233845         False         False  False           False         False   
233846         False         False  False           False         False   
233847         False         False  False           False         False   
233848         False         False  False           False         False   

        incidentmonth  maintenancerailroadcode  maintenancerailroadname  \
2529            False   

In [25]:
from tabulate import tabulate
print(tabulate(Raw_2023_clean, headers = 'keys', tablefmt = 'psql'))

NameError: name 'Raw_2023_clean' is not defined